# 02 — Carga e inspección

La primera celda de cualquier notebook de análisis hace estas dos cosas: cargar los datos e inspeccionarlos. Inspeccionar no es leer los datos visualmente, es ejecutar funciones que producen un diagnóstico estructurado.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'


## read_csv — parámetros esenciales

La mayoría de datasets del mundo real requieren al menos un parámetro extra en `read_csv`. Los defaults no siempre son los correctos.

In [ ]:
# Parámetros más usados en la práctica
df = pd.read_csv(
    TRAIN,
    # encoding='utf-8',          # codificación de caracteres (default)
    # encoding='latin-1',        # para archivos con caracteres especiales no UTF-8
    # sep=';',                   # separador (default ',')
    # decimal=',',               # decimal europeo
    # thousands='.',             # separador de miles
    low_memory=False,            # infiere tipos sobre el archivo completo, no por chunks
    # usecols=['Sales', 'Region'],  # cargar solo columnas necesarias
    # nrows=1000,                # cargar solo N filas (útil para exploración)
    # skiprows=1,                # saltar filas al inicio
    # parse_dates=['Order Date'], # convertir a datetime al cargar
    # dtype={'Postal Code': str}, # forzar tipo por columna
)
print(f'Cargado: {df.shape}')


In [ ]:
# Airbnb necesita latin-1 y tiene 33 columnas — ejemplo de caso real
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)
print(f'Airbnb: {air.shape}')
print(f'Memoria: {air.memory_usage(deep=True).sum() / 1024**2:.1f} MB')


## info() — el diagnóstico más completo

`info()` muestra en una sola llamada: número de filas, tipos de cada columna, y cuántos valores no nulos tiene cada una. Es la primera función que se ejecuta siempre.

In [ ]:
df.info()


In [ ]:
# Para datasets grandes, show_counts=True asegura que cuente los no-nulos
# memory_usage='deep' calcula el uso real de memoria (más lento pero preciso)
air.info(memory_usage='deep', show_counts=True)


## describe() — estadísticas descriptivas

In [ ]:
# Por defecto solo incluye columnas numéricas
print(df.describe().round(2))
print()

# include='object' para columnas de texto
print(df.describe(include='object'))


## value_counts, nunique, y sample

In [ ]:
# value_counts — frecuencia de cada valor, ordenada de mayor a menor
print(df['Region'].value_counts())
print()

# normalize=True para proporciones en vez de conteos
print(df['Category'].value_counts(normalize=True).round(3))
print()

# nunique — número de valores únicos por columna
print(df.nunique().sort_values())


In [ ]:
# sample() — muestra aleatoria, útil para inspeccionar filas reales
print(df.sample(5, random_state=42))
print()

# head/tail
print(df.tail(3))


## Diagnóstico de nulos

In [ ]:
# isnull().sum() — conteo de nulos por columna
nulos = df.isnull().sum()
print('Columnas con nulos:')
print(nulos[nulos > 0])
print()

# Porcentaje — dividir entre el total de filas
pct_nulos = (df.isnull().mean() * 100).round(2)
print('Porcentaje de nulos:')
print(pct_nulos[pct_nulos > 0])


---
## Resumen

| Función | Qué muestra |
|---------|-------------|
| `df.info()` | Tipos, non-null counts, memoria |
| `df.describe()` | Estadísticas numéricas |
| `df.describe(include='object')` | Stats para columnas de texto |
| `df['col'].value_counts()` | Frecuencia de cada valor |
| `df.nunique()` | Valores únicos por columna |
| `df.sample(n)` | n filas aleatorias |
| `df.isnull().sum()` | Nulos por columna |
